In [6]:
import pandas as pd
import folium

In [7]:
df = pd.read_html('../data/listado_iiee_peru.xls')

In [8]:
df = df[0]

In [9]:
df.head()

,Código Institución,Código Modular,Anexo,Nombre de SS.EE.,Ubigeo,Departamento,Provincia,Distrito,Código DRE/UGEL,DRE / UGEL,Centro Poblado,Código Centro Poblado,Código Local,Dirección,Nivel / Modalidad,Gestion / Dependencia,Latitud,Longitud,Altitud,Fuente de coordenadas
0,NaN,3994635,0,PEQUEÑOS ANGELES,40101,AREQUIPA,AREQUIPA,AREQUIPA,40001,AREQUIPA NORTE,7 DE JUNIO,557772,105451.0,CALLE FRANCISCO BOLOGNESI S/N,Inicial No Escolarizado,Pública - Sector Educación,-16.440842,-71.577918,2236,MED_REG (LOCAL)
1,22281046.0,1752989,0,VIGOUSKY SCHOOL,40101,AREQUIPA,AREQUIPA,AREQUIPA,40001,AREQUIPA NORTE,ABRAHAM MANRIQUE,570304,826398.0,ABRAHAM MANRIQUE MZ A LOTE 4,Inicial - Jardín,Privada - Particular,-16.420220,-71.540420,2303,UBICACION_WEB (LOCAL)
2,26198232.0,1237908,0,ALAS PERUANAS,40101,AREQUIPA,AREQUIPA,AREQUIPA,40001,AREQUIPA NORTE,APROVIORD,574202,57044.0,APROVIORD MZ B LOTE 6,Inicial - Jardín,Privada - Particular,-16.411600,-71.524800,2351,UGEL_GPS (LOCAL)
3,26198232.0,1030410,0,ALAS PERUANAS,40101,AREQUIPA,AREQUIPA,AREQUIPA,40001,AREQUIPA NORTE,APROVIORD,574202,57044.0,APROVIORD MZ B LOTE 6,Secundaria,Privada - Particular,-16.411600,-71.524800,2351,UGEL_GPS (LOCAL)
4,26198232.0,1030402,0,ALAS PERUANAS,40101,AREQUIPA,AREQUIPA,AREQUIPA,40001,AREQUIPA NORTE,APROVIORD,574202,57044.0,APROVIORD MZ B LOTE 6,Primaria,Privada - Particular,-16.411600,-71.524800,2351,UGEL_GPS (LOCAL)


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5068 entries, 0 to 5067
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Código Institución     3851 non-null   float64
 1   Código Modular         5068 non-null   int64  
 2   Anexo                  5068 non-null   str    
 3   Nombre de SS.EE.       5068 non-null   str    
 4   Ubigeo                 5068 non-null   int64  
 5   Departamento           5068 non-null   str    
 6   Provincia              5068 non-null   str    
 7   Distrito               5068 non-null   str    
 8   Código DRE/UGEL        5068 non-null   int64  
 9   DRE / UGEL             5068 non-null   str    
 10  Centro Poblado         5068 non-null   str    
 11  Código Centro Poblado  5068 non-null   int64  
 12  Código Local           5061 non-null   float64
 13  Dirección              5063 non-null   str    
 14  Nivel / Modalidad      5068 non-null   str    
 15  Gestion / Depen

In [11]:
df_lat_long = df[['Latitud', 'Longitud']]
LAT_COL='Latitud'
LON_COL='Longitud'

In [12]:
mapa = folium.Map(location=[-9.19, -75.01], zoom_start=6)

In [13]:
for _, row in df_lat_long.iterrows():
    folium.CircleMarker(
        location=[row[LAT_COL], row[LON_COL]],
        radius=3,
        color="blue",
        fill=True,
        popup=row.get("nombre", "Escuela")
    ).add_to(mapa)

mapa.save("escuelas.html")

In [14]:
df_lat_long.head()

,Latitud,Longitud
0,-16.440842,-71.577918
1,-16.420220,-71.540420
2,-16.411600,-71.524800
3,-16.411600,-71.524800
4,-16.411600,-71.524800


In [15]:
from geopy.distance import geodesic

dist = geodesic((df_lat_long.iloc[0][LAT_COL], df_lat_long.iloc[0][LON_COL]), (df_lat_long.iloc[1][LAT_COL], df_lat_long.iloc[1][LON_COL])).km
print(f"{dist} km")

4.609439398113379 km


In [16]:
import pandas as pd
from geopy.distance import geodesic

# Punto central y radio
punto_central = (df_lat_long.iloc[0][LAT_COL], df_lat_long.iloc[0][LON_COL])  # Arequipa, por ejemplo
radio_km = 10

def dentro_del_radio(row):
    coord = (row["Latitud"], row["Longitud"])
    return geodesic(punto_central, coord).km <= radio_km

escuelas_cercanas = df[df.apply(dentro_del_radio, axis=1)]
print(f"{len(escuelas_cercanas)} escuelas dentro de {radio_km} km")

2320 escuelas dentro de 10 km


In [17]:
import folium

mapa = folium.Map(location=list(punto_central), zoom_start=12)

# Dibujar el radio
folium.Circle(
    location=punto_central,
    radius=radio_km * 1000,  # en metros
    color="blue", fill=True, fill_opacity=0.1
).add_to(mapa)

# Punto central
folium.Marker(punto_central, popup="Centro", icon=folium.Icon(color="red")).add_to(mapa)

# Escuelas dentro del radio
for _, row in escuelas_cercanas.iterrows():
    folium.CircleMarker(
        location=[row["Latitud"], row["Longitud"]],
        radius=4, color="green", fill=True,
        popup=row.get("nombre", "Escuela")
    ).add_to(mapa)

mapa.save("radio_escuelas.html")